# Khôi phục cụm từ ASR bằng BERT — thử nghiệm

Ví dụ phát triển do người dùng cung cấp. Câu đích chỉ dùng **sau inference** để so sánh, không đi vào model. Không train, không API, không audio. Đây là suy đoán từ văn bản, không chứng minh người nói đã phát âm đúng câu đó. Chưa bật trong demo.

Thuật toán: tìm span bất thường → BERT đề xuất từ bên phải rồi dự đoán từ bên trái → chấm các ứng viên cùng cách bằng whole-word pseudo log likelihood → kiểm tra giới hạn thay đổi. Điểm và ngưỡng là thiết lập thử nghiệm, chưa hiệu chỉnh bằng validation độc lập.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
import torch
torch.set_num_threads(2)
try: torch.set_num_interop_threads(2)
except RuntimeError: pass
from scripts.probe_bert_phrase_recovery import PhraseProbe, VERSION
OUT = ROOT / 'data/generated/bert-phrase-recovery-v1'
OUT.mkdir(parents=True, exist_ok=True)
probe = PhraseProbe(ROOT)
print('Model: google-bert/bert-base-uncased | CPU | pretrained, không fine-tune')

Model: google-bert/bert-base-uncased | CPU | pretrained, không fine-tune


In [2]:
text = 'I will wi to order a cupf coffee and one ice ta please'
t0 = time.monotonic()
result = probe.recover(text)
elapsed = time.monotonic() - t0
print('Input :', text)
print('Output:', result['text'])
print('Thời gian gồm tải model:', round(elapsed, 3), 'giây')
for row in result['rounds']:
    for edit in row['evidence']:
        if edit['accepted']:
            print(f"Vòng {row['round']}: {edit['original']} → {edit['selected']}")
record = {'version': VERSION, 'input': text, 'result': result, 'elapsed_sec': elapsed,
          'target_supplied': False, 'no_audio': True, 'no_api': True, 'no_training': True,
          'production_enabled': False,
          'source_sha256': hashlib.sha256((ROOT/'scripts/probe_bert_phrase_recovery.py').read_bytes()).hexdigest()}
(OUT/'notebook-example.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')

Some weights of the model checkpoint at D:\NCC\MDD\models\pretrained\bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Input : I will wi to order a cupf coffee and one ice ta please
Output: I would like to order a cup of coffee and one ice tea please
Thời gian gồm tải model: 9.297 giây
Vòng 1: cupf → cup of
Vòng 1: ta → tea
Vòng 2: will wi → would like


4996

In [3]:
# Đáp án chỉ được dùng để đánh giá sau khi model đã trả kết quả.
expected = 'I would like to order a cup of coffee and one ice tea please'
print('Khớp câu mong muốn:', result['text'].casefold() == expected.casefold())
print('Từ chưa rõ:', result['unresolved_words'])
print('Đây là kiểm thử ví dụ phát triển, không phải accuracy trên tập test.')

Khớp câu mong muốn: True
Từ chưa rõ: []
Đây là kiểm thử ví dụ phát triển, không phải accuracy trên tập test.


In [4]:
controls = [
    ('clear_sentence', 'I would like to order a cup of coffee and one ice tea please'),
    ('alternative_intent', 'I will try to order a cupf coffee and one ice ta please'),
    ('negation_quantity', 'I do not want two coffees I want one tea'),
    ('negation_noisy', 'I do not want two coffees I want one cupf tea'),
    ('repetition', 'I I would like a coffee please'),
    ('learner_grammar', 'Yesterday I go to work'),
    ('short_answer', 'Yes'),
    ('proper_name', 'I am Soo and I like tea'),
    ('unrelated_error', 'I want a bottl of water'),
    ('unknown_words', 'Flerm narg zibble'),
]
rows = []
for label, sentence in controls:
    t0 = time.monotonic()
    item = probe.recover(sentence)
    rows.append({'case': label, 'input': sentence, 'output': item['text'],
                 'changed': item['text'] != sentence, 'status': item['status'],
                 'unresolved': item['unresolved_words'],
                 'elapsed_sec': round(time.monotonic() - t0, 3), 'detail': item})
    print(label, ':', item['text'], '|', item['status'])
(OUT/'controls.json').write_text(json.dumps({'rows': rows, 'no_target_supplied_to_model': True,
    'development_controls_only': True, 'training': False, 'audio_inference': False,
    'source_sha256': hashlib.sha256((ROOT/'scripts/probe_bert_phrase_recovery.py').read_bytes()).hexdigest()},
    ensure_ascii=False, indent=2), encoding='utf-8')

clear_sentence : I would like to order a cup of coffee and one ice tea please | unchanged


alternative_intent : I will try to order a cup of coffee and one ice tea please | inferred
negation_quantity : I do not want two coffees I want one tea | unchanged


negation_noisy : I do not want two coffees I want one cup of tea | inferred
repetition : I I would like a coffee please | unchanged
learner_grammar : Yesterday I go to work | unchanged
short_answer : Yes | unchanged
proper_name : I am Soo and I like tea | unchanged


unrelated_error : I want a bottl of water | ambiguous


unknown_words : Flerm narg zibble | ambiguous


17621

In [5]:
import pandas as pd
pd.DataFrame([{key: row[key] for key in ('case', 'input', 'output', 'changed', 'status', 'elapsed_sec')}
              for row in rows])

,case,input,output,changed,status,elapsed_sec
0,clear_sentence,I would like to order a cup of coffee and one ...,I would like to order a cup of coffee and one ...,False,unchanged,0.000
1,alternative_intent,I will try to order a cupf coffee and one ice ...,I will try to order a cup of coffee and one ic...,True,inferred,3.656
2,negation_quantity,I do not want two coffees I want one tea,I do not want two coffees I want one tea,False,unchanged,0.000
3,negation_noisy,I do not want two coffees I want one cupf tea,I do not want two coffees I want one cup of tea,True,inferred,2.328
4,repetition,I I would like a coffee please,I I would like a coffee please,False,unchanged,0.000
5,learner_grammar,Yesterday I go to work,Yesterday I go to work,False,unchanged,0.000
6,short_answer,Yes,Yes,False,unchanged,0.000
7,proper_name,I am Soo and I like tea,I am Soo and I like tea,False,unchanged,0.000
8,unrelated_error,I want a bottl of water,I want a bottl of water,False,ambiguous,1.844
9,unknown_words,Flerm narg zibble,Flerm narg zibble,False,ambiguous,5.375


Muốn chấm phát âm, cần đối chiếu audio với câu đã chọn; BERT chỉ nhận văn bản. Các sửa đổi vẫn là câu **suy đoán**. Một ví dụ đúng và các control này chưa đủ để bật tự động cho mọi đầu vào; cần đo thêm WER, thay đổi sai ý và thời gian trên dữ liệu độc lập.